# Attention Maps

Attention maps are visual representations that highlight the areas of an input image that a neural network focuses on when making predictions. They are particularly useful in understanding and interpreting the decision-making process of deep learning models in both classification and segmentation tasks.

In image classification tasks or segmentation, attention maps reveal which parts of an image are most influential in determining the class label. 

In this notebook we will see how to compute attention maps for the UNET model you trained in the "Segmentation and UNET" notebook (it loads the weights from `trained_unet.pth`, so run that notebook first). The goal of this notebook is to understand how attention maps can be computed and how they can be used to interpret the decision-making process of a neural network.

In [ ]:
# Some imports

import torch
import matplotlib.pyplot as plt

from cm2003.decathlon_dataset import get_decathlon_dataloader
from cm2003.unet import UNET
from cm2003.train import train

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')  # Apple-silicon Mac
else:
    device = torch.device('cpu')
print('using device:', device)

In [ ]:
# Load the data
root_dir = './cm2003/datasets'
task = "Task04_Hippocampus"

val_loader = get_decathlon_dataloader(root_dir, task, "validation", batch_size=4, num_workers=0)

# Load the model
model = UNET(in_channels=1, out_channels=3)

# Load the model weights saved in the "Segmentation and UNET" notebook
model.load_state_dict(torch.load("trained_unet.pth", map_location="cpu", weights_only=True))

# Set the model to evaluation mode (we compute the attention maps on the CPU)
model.eval()

## Understanding Attention maps and GradCAM

GradCAM (Gradient-weighted Class Activation Mapping) is a technique for visualizing the regions of an input image that a convolutional neural network focuses on when making predictions. Let's break down how it works:

1. **Target Layer Selection**: We choose a target layer in the network, typically one of the later convolutional layers that captures high-level features.

2. **Forward Pass**: The input image is passed through the network to obtain predictions.

3. **Backward Pass**: We perform backpropagation for the target class, computing gradients with respect to the target layer's activations.

4. **Importance Weighting**: The gradients are globally average pooled to obtain weights for each feature map in the target layer.

5. **Feature Map Weighting**: These weights are used to scale the corresponding feature maps, emphasizing the important features for the target class.

6. **Heatmap Creation**: The weighted feature maps are combined to create a heatmap, which is then passed through a ReLU, upsampled to the size of the input image and normalized.

Note that despite the name, these are not the attention weights of a transformer: Grad-CAM is a gradient-based way to see which regions a CNN relies on. For a segmentation model, we use the mean output of a class over all pixels as the "score" of that class.

By viewing these visualizations, we can gain insights into the model's decision-making process and identify potential biases or unexpected behaviors in our neural networks which can be particularly useful to understand the model's behaviour on medical images.

Let's use the model we trained in the previous notebook to make predictions on the validation set and compute the attention maps. We will use the `GradCAM` class to compute the attention maps, read through the `cm2003/attention_maps.py` file to understand how it works.
 

In [ ]:
from cm2003.attention_maps import GradCAM, apply_cam
# Use the model to generate attention maps
target_layer = model.downs[-1]  # The output of the last downsampling block (the deepest encoder features before the bottleneck)

grad_cam = GradCAM(model, target_layer)

# Find a validation slice that contains both parts of the hippocampus (classes 1 and 2)
sample_image = None
for sample_data in val_loader:
    for i in range(len(sample_data['label'])):
        label = sample_data['label'][i]
        if (label == 1).sum() > 10 and (label == 2).sum() > 10:
            sample_image = sample_data['image'][i:i+1]  # Keep the batch dimension: (1, 1, H, W)
            sample_label = sample_data['label'][i:i+1]
            break
    if sample_image is not None:
        break

# Generate prediction
with torch.no_grad():
    sample_prediction = model(sample_image).argmax(dim=1)

# Generate CAM for each class
for class_idx in range(3):  # 3 classes (background, anterior, posterior)
    cam = grad_cam.generate_cam(sample_image, class_idx)
    print(f"Attention map for class {class_idx}")
    apply_cam(sample_image[0], sample_label, cam, class_idx, sample_prediction)

grad_cam.remove_hooks()

## Question

What do you observe from the attention maps in this example? How do they change if you use another target layer, for example `model.bottleneck` or one of the decoder blocks in `model.ups`?


Try to plot the attention maps for the other samples in the validation set. If possible, try to find a sample that the model made a suboptimal/wrong prediction on and see if the attention maps are able to highlight the regions that the model focused on to make the wrong prediction.

In [ ]:
# Your code here